# FraudShield – Credit Card Fraud Detection System

In [1]:
!pip install imbalanced-learn==0.10.1 --no-deps


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.0/226.0 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: imbalanced-learn
    Found existing installation: imbalanced-learn 0.14.1
    Uninstalling imbalanced-learn-0.14.1:
      Successfully uninstalled imbalanced-learn-0.14.1


In [2]:
# Import the libraries
import numpy as np
import pandas as pd

# Importing visualization libraries
import matplotlib.pyplot as plt

# Model & metrics
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

# Imbalance handling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline



ImportError: cannot import name '_MissingValues' from 'sklearn.utils._param_validation' (/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py)

## 1.  Data Acquisition & Loading

In [ ]:
# Load dataset  and create a dataframe
df = pd.read_csv("/kaggle/input/creditcardfraud/creditcard.csv")


In [ ]:
print("First five rows of the dataset:")
df.head(5)

In [ ]:
print("\nMissing values:\n", df.isna().sum())


In [ ]:
print("\nClass distribution:\n", df['Class'].value_counts())

In [ ]:
print("Dimensions of the data frame:", df.shape)

In [ ]:
print("Dataset Info:")
print(df.info())

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Class distribution
class_counts = df['Class'].value_counts().sort_index()

plt.bar(class_counts.index, class_counts.values, color=['skyblue','red'])
plt.title("Class Distribution (0=Non-Fraud, 1=Fraud)")
plt.xlabel("Class")
plt.ylabel("Count")

for i, v in enumerate(class_counts.values):
    plt.text(i, v + (0.01 * v), str(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.show()


In [ ]:
# Time distribution
plt.hist(df['Time'], bins=50)
plt.title("Distribution of Time")
plt.xlabel("Time (seconds)")
plt.ylabel("Frequency")
plt.show()

# Amount distribution
plt.hist(df['Amount'], bins=50)
plt.title("Distribution of Amount")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Amount by Class
plt.hist(df.loc[df.Class==0, 'Amount'], bins=50, alpha=0.6, label="Non-Fraud")
plt.hist(df.loc[df.Class==1, 'Amount'], bins=50, alpha=0.6, label="Fraud")
plt.title("Amount by Class")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.legend()
plt.show()


In [ ]:
# Scatter plot: Time vs Amount by Class
plt.figure(figsize=(10,6))

# Non-fraud
plt.scatter(df[df['Class']==0]['Time'],
            df[df['Class']==0]['Amount'],
            c='skyblue', alpha=0.3, s=10, label="Non-Fraud")

# Fraud
plt.scatter(df[df['Class']==1]['Time'],
            df[df['Class']==1]['Amount'],
            c='red', alpha=0.6, s=20, label="Fraud")

plt.title("Transaction Distribution by Time & Amount")
plt.xlabel("Time (seconds since first transaction)")
plt.ylabel("Amount ($)")
plt.legend(loc="upper right")
plt.ylim(0, 3000)   # limit y-axis to see patterns (frauds are usually low amounts)
plt.show()


## 3. Data Preparation

In [ ]:
# Features / Target
X = df.drop(columns=['Class'])
y = df['Class']

# Train-Test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Fraud ratio in train:", y_train.mean())


In [ ]:
# Scale only 'Time' and 'Amount'
scale_cols = ['Time', 'Amount']

preprocess = ColumnTransformer(
    transformers=[('scale_ta', StandardScaler(), scale_cols)],
    remainder='passthrough'  # keep the other columns unchanged
)


## 4. Model Development and Evaluation

In [ ]:
# Logistic Regression
log_reg = LogisticRegression(max_iter=8000, solver="saga", class_weight="balanced", random_state=42)
log_reg.fit(X_train, y_train)

# Random Forest
rf_clf = RandomForestClassifier(n_estimators=200, class_weight="balanced",random_state=42)
rf_clf.fit(X_train, y_train)

# XGBoost
xgb_clf = XGBClassifier(n_estimators=200, eval_metric="logloss",
                        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train),random_state=42 )
xgb_clf.fit(X_train, y_train)

# Model Evaluation

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Reports
    print(f"\n===== {name} =====")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

    # Plots
    fig, ax = plt.subplots(figsize=(6, 5))  # just one axis
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax)
    ax.set_title(f"{name} - ROC Curve")
    plt.show()

# Evaluate all models
evaluate_model("Logistic Regression", log_reg, X_test, y_test)
evaluate_model("Random Forest", rf_clf, X_test, y_test)
evaluate_model("XGBoost", xgb_clf, X_test, y_test)

